# 00. 2021~2025 원본 데이터 인벤토리

파일·행·스키마·분기 커버리지와 연도별 코드 호환성을 확인한다. 추정매출은 다운로드 여부와 관계없이 `data/raw/sales` 파일을 사용한다.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.data.reporting import validate_raw_datasets
from src.data.historical import check_historical_compatibility
pd.set_option('display.max_columns', 100)
PROJECT_ROOT

## 데이터셋·파일·행·키 인벤토리

In [ ]:
raw_reports = validate_raw_datasets(project_root=PROJECT_ROOT, save=True)
inventory = raw_reports['inventory']
inventory[inventory['file'].eq('__TOTAL__')][['dataset','status','row_count','area_count','industry_count','quarter_count','quarter_min','quarter_max','missing_quarters']]

## 2021Q1~2025Q4 분기 커버리지와 누락 분기

In [ ]:
coverage = raw_reports['quarter_coverage']
display(coverage.pivot(index='dataset', columns='quarter', values='present').fillna(False))
coverage.assign(year=coverage['quarter'].astype(str).str[:4]).groupby(['dataset','year'], as_index=False)['row_count'].sum()

## 주요 컬럼·결측률·완전 중복

In [ ]:
schema = raw_reports['schema']
display(schema.sort_values('missing_rate', ascending=False).groupby('dataset').head(10))
display(raw_reports['duplicates'])

## 연도별 스키마·행 분포·상권코드 유지율·업종코드 교집합

In [ ]:
history = check_historical_compatibility(project_root=PROJECT_ROOT, save=True)
display(history['schema_changes'])
display(history['area_code_overlap'])
display(history['industry_code_overlap'])
display(history['yearly_distribution'])